# SSL Universal Training Notebook — Colab Pro+ (L4) + VSCode

**사용법**: Cell 3의 `GITHUB_URL` / `BRANCH` 변수와, Cell 4의 `CONFIG` 변수만 바꿔서 같은 노트북으로 모든 method 학습.

- `configs/mocov3_vits.yaml`   — **★ MoCo v3 + ViT-S/16 (현재 메인)**
- `configs/mocov2_mc_r50.yaml` — Track A (multi-crop MoCo, 비교)
- `configs/vicreg_r50.yaml`    — Track B (VICReg, 1차 끝난 뒤)
- `configs/mocov2_r50.yaml`    — 기존 baseline 재현용

**Drive 디렉토리 (1회 사전 생성)**:
```
내 드라이브/ssl_project/
├── data/      # STL10/CIFAR10 raw
├── outputs/   # checkpoints
├── logs/      # training logs
└── features/  # extracted (N, D) feature npy
```

**Pro+ background execution**: 우상단 메뉴 → 백그라운드 실행 ON. 노트북 닫아도 24h 유지.

**워크플로**: VSCode 편집 → `git push` to GitHub 브랜치 → Colab Cell 3 재실행으로 `git pull`. 코드는 Drive에 안 둠 (GitHub가 single source of truth).

In [1]:
# Cell 1 — GPU 확인
import torch, subprocess
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
if torch.cuda.is_available():
    print('VRAM:', torch.cuda.get_device_properties(0).total_memory/1e9, 'GB')
    print('BF16 OK:', torch.cuda.is_bf16_supported())
print(subprocess.check_output(['nvidia-smi', '-L']).decode())
# L4 22GB + bf16 OK 확인. T4면 disconnect 후 Runtime → Change runtime type → L4 재선택.

GPU: NVIDIA L4
VRAM: 23.65915136 GB
BF16 OK: True
GPU 0: NVIDIA L4 (UUID: GPU-f2b8416a-c83e-dcce-507c-ef0ae1bdfab3)



In [2]:
# Cell 2 — Google Drive mount
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Cell 3 — GitHub repo clone + install + Drive symlinks
#
# ★ 학습 전에 아래 두 변수만 본인 환경에 맞게 수정.
# ★ Private repo면 GITHUB_TOKEN을 Colab Secrets에 등록해서 사용 (아래 주석 참조).

GITHUB_URL = 'https://github.com/minsungk02/Visual-Intelligence-Learning-SSL-DINOFORCE-TV.git'
BRANCH     = 'main'

# ===== Private repo 인증이 필요한 경우 =====
# from google.colab import userdata
# token = userdata.get('GITHUB_TOKEN')   # Colab Secrets에 GITHUB_TOKEN 등록 후 사용
# GITHUB_URL = GITHUB_URL.replace('https://', f'https://{token}@')
# ===========================================

import os, subprocess
os.chdir('/content')

CLONE_DIR = '/content/ssl_project'
if not os.path.exists(CLONE_DIR):
    # 첫 clone — 지정한 브랜치 바로 받아오기
    subprocess.run(['git', 'clone', '-b', BRANCH, GITHUB_URL, CLONE_DIR], check=True)
os.chdir(CLONE_DIR)

# 매 세션마다 fetch + checkout + pull — 최신 코드 동기화
subprocess.run(['git', 'fetch', 'origin'], check=True)
subprocess.run(['git', 'checkout', BRANCH], check=True)
subprocess.run(['git', 'pull', 'origin', BRANCH], check=True)

# 어느 commit에서 학습하는지 기록 — RESULTS.md에 옮겨 적기
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD']).decode().strip()
print(f'Repo:   {GITHUB_URL}')
print(f'Branch: {BRANCH}')
print(f'Commit: {commit}')

# Editable install
subprocess.run(['pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
subprocess.run(['pip', 'install', '-q', '-e', '.'], check=True)

# Drive symlinks — data/outputs/logs/features는 Drive에 영구 저장 (코드는 GitHub)
DRIVE = '/content/drive/MyDrive/ssl_project'
for sub in ['data', 'outputs', 'logs', 'features']:
    os.makedirs(f'{DRIVE}/{sub}', exist_ok=True)
    local = f'/content/ssl_project/{sub}'
    if os.path.islink(local) or os.path.exists(local):
        subprocess.run(['rm', '-rf', local])
    os.symlink(f'{DRIVE}/{sub}', local)

print('Setup complete.')
subprocess.run(['ls', '-la', '/content/ssl_project'])

Repo:   https://github.com/minsungk02/Visual-Intelligence-Learning-SSL-DINOFORCE-TV.git
Branch: main
Commit: 5dbbbfef58f532d1a66ac7cdaf2a4dcba18cca2a
Setup complete.


CompletedProcess(args=['ls', '-la', '/content/ssl_project'], returncode=0)

In [ ]:
# Cell 4 — 학습 시작
#
# ★ 아래 CONFIG 한 줄만 바꾸면 OUT_KEY / Drive 저장경로 / (Cell 5)추출경로가 전부 자동 연동됩니다.
import glob, subprocess, os, yaml

CONFIG = 'configs/mocov3_vits8.yaml'    # ⭐ ViT-S/8 (patch8, fine-grained 실험)
# CONFIG = 'configs/mocov3_vits.yaml'   # ViT-S/16 (이전 메인, STL10/CIFAR10 86.6%)
# CONFIG = 'configs/mocov2_mc_r34.yaml' # R34 multi-crop
# CONFIG = 'configs/vicreg_r50.yaml'    # VICReg

# OUT_KEY를 config의 output.dir에서 자동 유도 (경로 불일치/덮어쓰기 방지)
OUT_KEY = yaml.safe_load(open(f'/content/ssl_project/{CONFIG}'))['output']['dir'].replace('./', '')
print(f'CONFIG  = {CONFIG}')
print(f'OUT_KEY = {OUT_KEY}   (Drive 저장: MyDrive/ssl_project/{OUT_KEY})')

SANITY_EPOCHS = 3   # 첫 실행: 3 (메모리/시간 확인). 풀학습 진입 시: None

# 자동 resume — Drive에 마지막 ckpt 있으면 이어서 시작
ckpts = sorted(glob.glob(f'/content/ssl_project/{OUT_KEY}/ckpt_ep*.pth'),
               key=lambda p: int(p.rsplit('_ep', 1)[1].split('.')[0]))
resume_args = ['--resume', ckpts[-1]] if ckpts else []
print('Resume:', ckpts[-1] if ckpts else 'None (fresh start)')

extra = ['--epochs', str(SANITY_EPOCHS)] if SANITY_EPOCHS is not None else []
if SANITY_EPOCHS is not None:
    print(f'>>> SANITY MODE — {SANITY_EPOCHS} epochs only')

cmd = ['python', '-u', 'scripts/pretrain.py', '--config', CONFIG] + resume_args + extra
print('CMD:', ' '.join(cmd))
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        bufsize=1, text=True, cwd='/content/ssl_project')
for line in proc.stdout:
    print(line, end='')
proc.wait()
print('Exit code:', proc.returncode)

In [ ]:
# Cell 5 — Feature 추출 (Cell 4의 CONFIG/OUT_KEY 자동 연동)
import subprocess, os, glob, yaml

# Cell 4에서 정한 CONFIG/OUT_KEY 재사용 (Cell 5만 단독 실행 시 여기서 지정)
try:
    CONFIG, OUT_KEY
except NameError:
    CONFIG = 'configs/mocov3_vits8.yaml'
    OUT_KEY = yaml.safe_load(open(f'/content/ssl_project/{CONFIG}'))['output']['dir'].replace('./', '')

# ★ feature 추출 방식 (Phase A LP 레버, ViT 전용). 채택값 = last4_cls_patchmean.
#   cls(384, baseline) / avg(384) / cls_patchmean(768) / last4_cls(1536, DINO표준) /
#   last4_cls_patchmean(1920, ★채택). ResNet/Swin이면 무시되고 cls로 추출됨.
#   5종 sweep을 원하면 MODES=['cls',...] 로 두고 이 셀 본문을 for문으로 감싸면 됨.
FEATURE_MODE = 'last4_cls_patchmean'

OUT_DIR = f'/content/ssl_project/{OUT_KEY}'
TAG = OUT_KEY.split('/')[-1].replace('_seed42', '')   # 예: mocov3_vits8
ckpts = sorted(glob.glob(f'{OUT_DIR}/backbone_ep*.pth'),
               key=lambda p: int(p.rsplit('_ep', 1)[1].split('.')[0]))
assert ckpts, f'backbone_ep*.pth 없음: {OUT_DIR} (학습 먼저)'
BACKBONE = ckpts[-1]
EP = BACKBONE.rsplit('_ep', 1)[1].split('.')[0]
CONFIG_PATH = f'/content/ssl_project/{CONFIG}'
# mode를 경로에 포함 → 모드별 feature가 서로 덮어쓰지 않음 (Cell 6/7이 이 FEAT_DIR 재사용)
FEAT_DIR = f'/content/drive/MyDrive/ssl_project/features/{TAG}_ep{EP}_{FEATURE_MODE}'
print(f'backbone:     {BACKBONE}  (epoch {EP})')
print(f'feature_mode: {FEATURE_MODE}')
print(f'FEAT_DIR:     {FEAT_DIR}')

NORMALIZE = 'standardize'   # {'none','l2','standardize'}
os.makedirs(FEAT_DIR, exist_ok=True)
cmd = ['python', '-u', 'scripts/extract_features.py',
       '--backbone', BACKBONE, '--config', CONFIG_PATH,
       '--output-dir', FEAT_DIR, '--normalize', NORMALIZE,
       '--feature-mode', FEATURE_MODE, '--batch-size', '256']
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        bufsize=1, text=True, cwd='/content/ssl_project')
for line in proc.stdout:
    print(line, end='')
proc.wait()
print(f'\nexit {proc.returncode}')
print('Feature dir:', FEAT_DIR)

In [ ]:
# Cell 6 — LP 평가 (evaluate.py 고정 recipe — 수정 절대 금지)
import subprocess, glob, os

# Cell 5의 FEAT_DIR 재사용 (단독 실행 시 가장 최근 추출 폴더 자동 선택)
try:
    FEAT_DIR
except NameError:
    feat_dirs = sorted(glob.glob('/content/drive/MyDrive/ssl_project/features/*_ep*'),
                       key=os.path.getmtime)
    assert feat_dirs, 'features 폴더 없음 (Cell 5 먼저 실행)'
    FEAT_DIR = feat_dirs[-1]
print(f'Evaluating:   {FEAT_DIR}')
print(f'feature_mode: {globals().get("FEATURE_MODE", "(FEAT_DIR에서 추론)")}')
# evaluate.py LP recipe는 불변. feature 차원은 npy에서 자동 감지되므로
# last4_cls_patchmean(1920-d)도 추가 수정 없이 그대로 평가됨.

cmd = ['python', '-u', 'evaluate.py',
    '--stl10-train-features',   f'{FEAT_DIR}/stl10_train_features.npy',
    '--stl10-train-labels',     f'{FEAT_DIR}/stl10_train_labels.npy',
    '--stl10-test-features',    f'{FEAT_DIR}/stl10_test_features.npy',
    '--stl10-test-labels',      f'{FEAT_DIR}/stl10_test_labels.npy',
    '--cifar10-train-features', f'{FEAT_DIR}/cifar10_train_features.npy',
    '--cifar10-train-labels',   f'{FEAT_DIR}/cifar10_train_labels.npy',
    '--cifar10-test-features',  f'{FEAT_DIR}/cifar10_test_features.npy',
    '--cifar10-test-labels',    f'{FEAT_DIR}/cifar10_test_labels.npy',
    '--device', 'cuda']
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        bufsize=1, text=True, cwd='/content/ssl_project')
for line in proc.stdout:
    print(line, end='')
proc.wait()
print(f'\nexit {proc.returncode}')

In [ ]:
# Cell 7 — feature 시각화 (Cell 5의 FEAT_DIR 자동 연동)
#
# 경로 하드코딩 없음: Cell 5에서 추출한 FEAT_DIR을 그대로 사용.
# Cell 7만 단독 실행하면 가장 최근 추출 폴더(features/<tag>_ep<N>_<mode>)를 자동 선택.
# 차원 무관(_train_lp가 shape[1]로 D 자동) → last4_cls_patchmean(1920-d)도 그대로 동작.
import os, glob, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.metrics import confusion_matrix

try:
    FEAT_DIR
except NameError:
    feat_dirs = sorted(glob.glob('/content/drive/MyDrive/ssl_project/features/*_ep*'),
                       key=os.path.getmtime)
    assert feat_dirs, 'features 폴더 없음 (Cell 5 먼저 실행)'
    FEAT_DIR = feat_dirs[-1]
DATA = '/content/drive/MyDrive/ssl_project/data'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
MODE = globals().get('FEATURE_MODE', os.path.basename(FEAT_DIR).split('_ep')[-1])
print(f'Visualizing:  {FEAT_DIR}   (device {device})')
print(f'feature_mode: {MODE}')

STL10_CLASSES   = ['airplane','bird','car','cat','deer','dog','horse','monkey','ship','truck']
CIFAR10_CLASSES = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']

def _load(ds, split):
    x = np.load(f'{FEAT_DIR}/{ds}_{split}_features.npy').astype('float32')
    y = np.load(f'{FEAT_DIR}/{ds}_{split}_labels.npy').reshape(-1).astype('int64')
    return x, y

def _train_lp(tr_x, tr_y, te_x, epochs=100, bs=128, seed=42):
    # evaluate.py 동일 recipe: SGD lr0.1 mom0.9 wd0, cosine 100ep, bs128
    torch.manual_seed(seed)
    D, C = tr_x.shape[1], int(tr_y.max())+1
    head = nn.Linear(D, C).to(device)
    opt = torch.optim.SGD(head.parameters(), lr=0.1, momentum=0.9, weight_decay=0.0)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-6)
    Xtr = torch.tensor(tr_x, device=device); Ytr = torch.tensor(tr_y, device=device)
    n = Xtr.shape[0]
    for ep in range(epochs):
        head.train(); perm = torch.randperm(n, device=device)
        for i in range(0, n, bs):
            idx = perm[i:i+bs]
            loss = F.cross_entropy(head(Xtr[idx]), Ytr[idx])
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        sch.step()
    head.eval()
    with torch.no_grad():
        return head(torch.tensor(te_x, device=device)).argmax(1).cpu().numpy()

def visualize(ds, classes, show_images=True, tsne_n=3000):
    try:
        tr_x, tr_y = _load(ds, 'train'); te_x, te_y = _load(ds, 'test')
    except FileNotFoundError:
        print(f'[{ds}] feature 없음 — 스킵'); return
    print(f'\n===== {ds.upper()} [{MODE}] =====  train {tr_x.shape}, test {te_x.shape}')
    pred = _train_lp(tr_x, tr_y, te_x)
    acc = (pred==te_y).mean()*100
    print(f'재현 Top-1: {acc:.2f}%')

    # 1) t-SNE
    n = min(tsne_n, te_x.shape[0])
    sel = np.random.RandomState(42).choice(te_x.shape[0], n, replace=False)
    emb = TSNE(n_components=2, init='pca', perplexity=30, random_state=42).fit_transform(te_x[sel])
    plt.figure(figsize=(9,8))
    for c in range(10):
        m = te_y[sel]==c; plt.scatter(emb[m,0], emb[m,1], s=6, alpha=0.6, label=classes[c])
    plt.legend(markerscale=2, fontsize=9); plt.title(f'{ds} [{MODE}] t-SNE (Top-1 {acc:.1f}%)'); plt.show()

    # 2) confusion matrix
    cm = confusion_matrix(te_y, pred); cmn = cm/cm.sum(1, keepdims=True).clip(min=1)
    plt.figure(figsize=(8,7)); plt.imshow(cmn, cmap='Blues', vmin=0, vmax=1); plt.colorbar(fraction=0.046)
    plt.xticks(range(10), classes, rotation=45, ha='right'); plt.yticks(range(10), classes)
    for i in range(10):
        for j in range(10):
            if cmn[i,j]>0.01:
                plt.text(j,i,f'{cmn[i,j]:.2f}',ha='center',va='center',
                         color='white' if cmn[i,j]>0.5 else 'black', fontsize=7)
    plt.ylabel('True'); plt.xlabel('Pred'); plt.title(f'{ds} confusion (acc {acc:.1f}%)'); plt.show()

    # 3) per-class accuracy
    pc = [(pred[te_y==c]==c).mean()*100 for c in range(10)]
    plt.figure(figsize=(9,4)); bars = plt.bar(range(10), pc, color='steelblue')
    plt.xticks(range(10), classes, rotation=45, ha='right'); plt.ylim(0,100)
    plt.axhline(acc, color='red', ls='--', label=f'overall {acc:.1f}%')
    for b,v in zip(bars,pc): plt.text(b.get_x()+b.get_width()/2, v+1, f'{v:.0f}', ha='center', fontsize=8)
    plt.legend(); plt.ylabel('%'); plt.title(f'{ds} per-class accuracy'); plt.show()

    # 4) 샘플 예측 (실제 이미지)
    if show_images:
        try:
            from torchvision import datasets
            d = datasets.STL10(DATA, split='test', download=True) if ds=='stl10' \
                else datasets.CIFAR10(DATA, train=False, download=True)
            cor = np.where(pred==te_y)[0]; wr = np.where(pred!=te_y)[0]
            r = np.random.RandomState(0)
            sel2 = list(r.choice(cor, 8, replace=False)) + list(r.choice(wr, min(8,len(wr)), replace=False))
            fig, axes = plt.subplots(4,4, figsize=(11,12))
            for ax, i in zip(axes.flat, sel2):
                img,_ = d[int(i)]; ax.imshow(img); ax.axis('off')
                ok = pred[i]==te_y[i]
                ax.set_title(f'T:{classes[te_y[i]]}\nP:{classes[pred[i]]}',
                             color='green' if ok else 'red', fontsize=9)
            for ax in axes.flat[len(sel2):]: ax.axis('off')
            plt.suptitle(f'{ds} [{MODE}] 예측 (위2줄 정답 / 아래2줄 오답)'); plt.tight_layout(); plt.show()
        except Exception as e:
            print(f'샘플 이미지 스킵: {e}')

visualize('stl10', STL10_CLASSES)
visualize('cifar10', CIFAR10_CLASSES)

## 운영 팁

- **MoCo v3 + ViT-S/16 (메인) / L4 / batch 1024**: config `backbone.gradient_checkpoint: true`가 activation을 depth배 줄여 24GB에 안전. 만약 그래도 OOM이면 config에서 `batch_size: 512` + `optimizer.lr: 3.0e-4` (lr = 1.5e-4 × batch/256). grad ckpt는 추출 시 자동 무시.
- **L4 22GB / multi-crop(2g+4l) / batch 256** (MoCo v2 트랙): 약 18-20GB 사용. OOM이면 `--batch-size 192` 또는 config의 `gc_mode`를 `"all"`로.
- **bf16 AMP** 활성 — fp16보다 안정, GradScaler 없음. `torch.cuda.is_bf16_supported()`가 True여야.
- **torch.compile**: 첫 epoch에 3-5분 컴파일 오버헤드. resume 시 캐시 무효화 가능 — 가능하면 fresh run을 한 번에.
- **save_every=5 + 자동 resume**: 세션 끊겨도 ≤ 5 epoch 손실. Cell 4가 Drive의 마지막 `ckpt_ep*.pth`를 자동 탐색해 `--resume`. optimizer(AdamW state)·momentum encoder·lr scheduler step까지 복원되어 끊긴 지점에서 그대로 이어짐 (smoke_test_mocov3.py [9]에서 검증).
- **시간 예산 추적**: 첫 10 epoch 평균 시간 × total_epochs로 추정 → 72h 초과 위험 시 epoch 줄여서 중단.
- **코드 업데이트 흐름**: VSCode 편집 → `git push origin minsung` → Colab Cell 3 재실행 (`git pull`이 자동) → Cell 4 재실행.
- **VSCode 운영**: 코드 편집은 VSCode + git push, 실행은 Colab 브라우저. 가장 안정적.